In [90]:
bronze_path = "Files/bronze/citybikes/bicing/bicing_snapshot.json"

raw_bicing_df = (
    spark.read
    .option("multiline", "true")
    .json(bronze_path)
)

raw_bicing_df.printSchema()

StatementMeta(, 9b0bc64d-9ad6-4ebe-b396-fb9ffb702d5a, 92, Finished, Available, Finished, False)

root
 |-- network: struct (nullable = true)
 |    |-- company: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ebikes: boolean (nullable = true)
 |    |-- href: string (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- location: struct (nullable = true)
 |    |    |-- city: string (nullable = true)
 |    |    |-- country: string (nullable = true)
 |    |    |-- latitude: double (nullable = true)
 |    |    |-- longitude: double (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- stations: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- empty_slots: long (nullable = true)
 |    |    |    |-- extra: struct (nullable = true)
 |    |    |    |    |-- ebikes: long (nullable = true)
 |    |    |    |    |-- has_ebikes: boolean (nullable = true)
 |    |    |    |    |-- normal_bikes: long (nullable = true)
 |    |    |    |    |-- online: boolean (nullable = true)
 |    |  

In [91]:
display(
    raw_bicing_df.select(
        "network.id",
        "network.name",
        "network.location",
        "network.stations"
    )
)

StatementMeta(, 9b0bc64d-9ad6-4ebe-b396-fb9ffb702d5a, 93, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5992f0ff-cff8-46a1-85b7-45cf263ada66)

In [92]:
from pyspark.sql.functions import explode, col

stations_df = (
    raw_bicing_df
    .select(
        explode(col("network.stations")).alias("station")
    )
)

bicing_df = stations_df.select("station.*")

display(bicing_df)

StatementMeta(, 9b0bc64d-9ad6-4ebe-b396-fb9ffb702d5a, 94, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d8597fbc-4165-4d1b-86f3-597a0148a150)

In [93]:
bicing_df.printSchema()

StatementMeta(, 9b0bc64d-9ad6-4ebe-b396-fb9ffb702d5a, 95, Finished, Available, Finished, False)

root
 |-- empty_slots: long (nullable = true)
 |-- extra: struct (nullable = true)
 |    |-- ebikes: long (nullable = true)
 |    |-- has_ebikes: boolean (nullable = true)
 |    |-- normal_bikes: long (nullable = true)
 |    |-- online: boolean (nullable = true)
 |    |-- uid: long (nullable = true)
 |-- free_bikes: long (nullable = true)
 |-- id: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- name: string (nullable = true)
 |-- timestamp: string (nullable = true)



In [94]:
display(
    bicing_df.select(
        "id",
        "name",
        "free_bikes",
        "empty_slots",
        "extra.*"
    )
)

StatementMeta(, 9b0bc64d-9ad6-4ebe-b396-fb9ffb702d5a, 96, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 70fe4b7c-7c21-41f2-8fe7-35bacb8e915b)

In [95]:
from pyspark.sql.functions import (
    col,
    current_timestamp,
    to_timestamp,
    trim, 
    regexp_replace
)

silver_bicing_df = (
    bicing_df
    .select(
        col("id").alias("station_id"),
        col("extra.uid").cast("long").alias("station_uid"),
        trim(col("name")).alias("station_name"),

        col("latitude").cast("double").alias("latitude"),
        col("longitude").cast("double").alias("longitude"),

        to_timestamp(regexp_replace(col("timestamp"), "Z$", ""),"yyyy-MM-dd'T'HH:mm:ss.SSSSSSXXX").alias("source_timestamp"),

        col("free_bikes").cast("long").alias("free_bikes"),
        col("empty_slots").cast("long").alias("empty_slots"),

        col("extra.ebikes").cast("long").alias("ebikes"),
        col("extra.normal_bikes").cast("long").alias("normal_bikes"),

        col("extra.has_ebikes").cast("boolean").alias("has_ebikes"),
        col("extra.online").cast("boolean").alias("is_online")
    )
    .withColumn(
        "station_capacity",
        col("free_bikes") + col("empty_slots")
    )
    .withColumn(
    "bike_breakdown_valid",
    col("free_bikes") == (col("ebikes") + col("normal_bikes"))
    )
    .withColumn(
        "silver_processed_at",
        current_timestamp()
    )
    .dropDuplicates(["station_id", "source_timestamp"])
)

display(silver_bicing_df)

StatementMeta(, 9b0bc64d-9ad6-4ebe-b396-fb9ffb702d5a, 97, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 872a08c0-25f3-4233-bf7e-1a77cb216c9c)

In [96]:
silver_bicing_df.printSchema()

print(f"Bicing Silver records: {silver_bicing_df.count()}")

StatementMeta(, 9b0bc64d-9ad6-4ebe-b396-fb9ffb702d5a, 98, Finished, Available, Finished, False)

root
 |-- station_id: string (nullable = true)
 |-- station_uid: long (nullable = true)
 |-- station_name: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- source_timestamp: timestamp (nullable = true)
 |-- free_bikes: long (nullable = true)
 |-- empty_slots: long (nullable = true)
 |-- ebikes: long (nullable = true)
 |-- normal_bikes: long (nullable = true)
 |-- has_ebikes: boolean (nullable = true)
 |-- is_online: boolean (nullable = true)
 |-- station_capacity: long (nullable = true)
 |-- bike_breakdown_valid: boolean (nullable = true)
 |-- silver_processed_at: timestamp (nullable = false)

Bicing Silver records: 544


In [97]:
total_rows = silver_bicing_df.count()

duplicate_station_snapshots = (
    silver_bicing_df
    .groupBy("station_id", "source_timestamp")
    .count()
    .filter(col("count") > 1)
    .count()
)

null_critical = silver_bicing_df.filter(
    col("station_id").isNull()
    | col("source_timestamp").isNull()
    | col("latitude").isNull()
    | col("longitude").isNull()
).count()

invalid_availability = silver_bicing_df.filter(
    (col("free_bikes") < 0)
    | (col("empty_slots") < 0)
    | (col("ebikes") < 0)
    | (col("normal_bikes") < 0)
).count()

invalid_coordinates = silver_bicing_df.filter(
    (col("latitude") < -90)
    | (col("latitude") > 90)
    | (col("longitude") < -180)
    | (col("longitude") > 180)
).count()

print(f"Total rows: {total_rows}")
print(f"Duplicate station snapshots: {duplicate_station_snapshots}")
print(f"Rows with critical nulls: {null_critical}")
print(f"Rows with invalid availability: {invalid_availability}")
print(f"Rows with invalid coordinates: {invalid_coordinates}")

StatementMeta(, 9b0bc64d-9ad6-4ebe-b396-fb9ffb702d5a, 99, Finished, Available, Finished, False)

Total rows: 544
Duplicate station snapshots: 0
Rows with critical nulls: 0
Rows with invalid availability: 0
Rows with invalid coordinates: 0


In [98]:
from pyspark.sql.functions import count, when

display(
    silver_bicing_df.select([
        count(
            when(col(c).isNull(), c)
        ).alias(c)
        for c in [
            "station_id",
            "source_timestamp",
            "latitude",
            "longitude"
        ]
    ])
)

StatementMeta(, 9b0bc64d-9ad6-4ebe-b396-fb9ffb702d5a, 100, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7d049e2f-7c91-4a47-b9b8-36d2ca63acb7)

In [99]:
null_availability = silver_bicing_df.filter(
    col("free_bikes").isNull()
    | col("empty_slots").isNull()
    | col("ebikes").isNull()
    | col("normal_bikes").isNull()
).count()

bike_breakdown_mismatch = silver_bicing_df.filter(
    col("free_bikes") != (col("ebikes") + col("normal_bikes"))
).count()

print(f"Rows with null availability values: {null_availability}")
print(f"Rows with inconsistent bike breakdown: {bike_breakdown_mismatch}")

StatementMeta(, 9b0bc64d-9ad6-4ebe-b396-fb9ffb702d5a, 101, Finished, Available, Finished, False)

Rows with null availability values: 0
Rows with inconsistent bike breakdown: 1


In [100]:
assert total_rows > 0, "Bicing Silver table is empty"

assert duplicate_station_snapshots == 0, \
    "Duplicate station snapshots detected"

assert null_critical == 0, \
    "Null values detected in critical station fields"

assert null_availability == 0, \
    "Null values detected in availability fields"

assert invalid_availability == 0, \
    "Negative availability values detected"

assert invalid_coordinates == 0, \
    "Invalid station coordinates detected"

if bike_breakdown_mismatch > 0:
    print(
        f"WARNING: {bike_breakdown_mismatch} station records "
        "have an inconsistent bike breakdown."
    )

print("All critical Bicing Silver data quality checks passed.")

StatementMeta(, 9b0bc64d-9ad6-4ebe-b396-fb9ffb702d5a, 102, Finished, Available, Finished, False)

All critical Bicing Silver data quality checks passed.


In [101]:
silver_table = "silver_bicing_station_status"

(
    silver_bicing_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

print(f"Delta table created: {silver_table}")

StatementMeta(, 9b0bc64d-9ad6-4ebe-b396-fb9ffb702d5a, 103, Finished, Available, Finished, False)

Delta table created: silver_bicing_station_status


In [102]:
validated_bicing_df = spark.table(
    "silver_bicing_station_status"
)

print(
    f"Rows in Bicing Silver Delta table: "
    f"{validated_bicing_df.count()}"
)

validated_bicing_df.printSchema()

display(validated_bicing_df)

StatementMeta(, 9b0bc64d-9ad6-4ebe-b396-fb9ffb702d5a, 104, Finished, Available, Finished, False)

Rows in Bicing Silver Delta table: 544
root
 |-- station_id: string (nullable = true)
 |-- station_uid: long (nullable = true)
 |-- station_name: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- source_timestamp: timestamp (nullable = true)
 |-- free_bikes: long (nullable = true)
 |-- empty_slots: long (nullable = true)
 |-- ebikes: long (nullable = true)
 |-- normal_bikes: long (nullable = true)
 |-- has_ebikes: boolean (nullable = true)
 |-- is_online: boolean (nullable = true)
 |-- station_capacity: long (nullable = true)
 |-- bike_breakdown_valid: boolean (nullable = true)
 |-- silver_processed_at: timestamp (nullable = true)



SynapseWidget(Synapse.DataFrame, 1de83f78-5c4d-4910-bf93-71fd6f0dad4b)